# 04. R analytics, hypothesis tests, ggplot2 charts and logistic regression

This is the R analytics section. The notebook does the tidyverse wrangling, builds 5 ggplot2 charts, runs 4 hypothesis tests, and fits a logistic regression of delivery failure on order and customer level features.

It needs the R kernel. The 9 CSVs are pulled from the GitHub repo at runtime, so this notebook is independent of notebook 03 and a marker can just click Run all from the top.

## Setup

In [ ]:
needed <- c("DBI", "RSQLite", "dplyr", "tidyr", "readr", "ggplot2", "scales")
missing <- setdiff(needed, rownames(installed.packages()))
if (length(missing) > 0) install.packages(missing, repos = "https://cloud.r-project.org")
suppressPackageStartupMessages({
  library(DBI); library(RSQLite); library(dplyr); library(tidyr)
  library(readr); library(ggplot2); library(scales)
})
set.seed(42)

In [ ]:
# Read the 9 NorthStar CSVs straight from the GitHub repo.
# The CSVs are committed under northstar_dataset/ on the main branch,
# so the marker does not have to upload anything before running the
# cell. Just press Run all from the top.

base_url <- "https://raw.githubusercontent.com/D1lxrry/Nortstar-Task/main/northstar_dataset"

customers  <- read.csv(file.path(base_url, "customers.csv"),  stringsAsFactors = FALSE)
orders     <- read.csv(file.path(base_url, "orders.csv"),     stringsAsFactors = FALSE)
deliveries <- read.csv(file.path(base_url, "deliveries.csv"), stringsAsFactors = FALSE)
drivers    <- read.csv(file.path(base_url, "drivers.csv"),    stringsAsFactors = FALSE)
vehicles   <- read.csv(file.path(base_url, "vehicles.csv"),   stringsAsFactors = FALSE)
hubs       <- read.csv(file.path(base_url, "hubs.csv"),       stringsAsFactors = FALSE)
incidents  <- read.csv(file.path(base_url, "incidents.csv"),  stringsAsFactors = FALSE)
complaints <- read.csv(file.path(base_url, "complaints.csv"), stringsAsFactors = FALSE)
app_events <- read.csv(file.path(base_url, "app_events.csv"), stringsAsFactors = FALSE)

cat("Loaded:", nrow(customers), "customers,", nrow(orders), "orders,",
    nrow(deliveries), "deliveries,", nrow(drivers), "drivers.\n")


## Build the analytical data frame

In [ ]:
complaint_counts <- complaints |> count(order_id, name = "complaint_count")
event_counts <- app_events |> filter(!is.na(order_id)) |> count(order_id, name = "app_event_count")
incident_counts <- incidents |> count(delivery_id, name = "incident_count")

dat <- orders |>
  left_join(deliveries, by = "order_id") |>
  left_join(customers, by = "customer_id") |>
  left_join(drivers, by = "driver_id") |>
  left_join(incident_counts, by = "delivery_id") |>
  left_join(complaint_counts, by = "order_id") |>
  left_join(event_counts, by = "order_id") |>
  mutate(
    incident_count = coalesce(incident_count, 0L),
    complaint_count = coalesce(complaint_count, 0L),
    app_event_count = coalesce(app_event_count, 0L),
    has_delivery = !is.na(delivery_id),
    failed = delivery_status == "Failed",
    delivered = !is.na(delivery_status) & delivery_status %in% c("OnTime", "Delayed", "Failed"),
    priority_level = factor(priority_level, levels = c("Low", "Medium", "High", "Critical")),
    service_type = factor(service_type),
    pickup_zone = factor(pickup_zone)
  )
cat("Analytical data frame:", nrow(dat), "rows,", ncol(dat), "cols\n")

## 5 ggplot visualisations

In [ ]:
# Chart 1. Revenue by service line.
dat |>
  group_by(service_type) |>
  summarise(revenue = sum(order_value, na.rm = TRUE), orders = n()) |>
  ggplot(aes(reorder(service_type, -revenue), revenue)) +
  geom_col(fill = "#1f77b4") +
  geom_text(aes(label = paste0(orders, " orders")), vjust = -0.4, size = 3.5) +
  scale_y_continuous(labels = label_comma(prefix = "GBP ")) +
  labs(title = "Revenue by service line", x = "Service type", y = "Total revenue") +
  theme_minimal()

In [ ]:
# Chart 2. Rating distribution by service line.
dat |>
  filter(!is.na(customer_rating_post_delivery)) |>
  ggplot(aes(customer_rating_post_delivery, fill = service_type)) +
  geom_histogram(binwidth = 0.5, colour = "white", alpha = 0.85) +
  facet_wrap(~ service_type, ncol = 3) +
  labs(title = "Customer rating distribution, by service line", x = "Rating", y = "Deliveries") +
  theme_minimal() + theme(legend.position = "none")

In [ ]:
# Chart 3. Failure heatmap zone x service.
dat |>
  filter(delivered) |>
  group_by(pickup_zone, service_type) |>
  summarise(rate = mean(failed, na.rm = TRUE), .groups = "drop") |>
  ggplot(aes(service_type, pickup_zone, fill = rate)) +
  geom_tile(colour = "white") +
  geom_text(aes(label = sprintf("%.0f%%", 100 * rate)), colour = "white", size = 3.4) +
  scale_fill_gradient(low = "#cce4f7", high = "#b22234", labels = label_percent()) +
  labs(title = "Delivery failure rate, zone x service", x = "Service", y = "Pickup zone") +
  theme_minimal()

In [ ]:
# Chart 4. Distance vs rating with loess smoothers per outcome.
dat |>
  filter(!is.na(route_distance_km), !is.na(customer_rating_post_delivery)) |>
  ggplot(aes(route_distance_km, customer_rating_post_delivery, colour = delivery_status)) +
  geom_point(alpha = 0.5, size = 1.2) +
  geom_smooth(method = "loess", se = FALSE) +
  scale_colour_manual(values = c(OnTime = "#2ca02c", Delayed = "#ff7f0e", Failed = "#d62728")) +
  labs(title = "Customer rating vs route distance",
       x = "Distance (km)", y = "Rating", colour = "Outcome") +
  theme_minimal()

In [ ]:
# Chart 5. Boxplot of rating by pickup_zone, ordered by median.
zone_order <- dat |>
  filter(!is.na(customer_rating_post_delivery)) |>
  group_by(pickup_zone) |>
  summarise(med = median(customer_rating_post_delivery)) |>
  arrange(med) |> pull(pickup_zone) |> as.character()

dat |>
  filter(!is.na(customer_rating_post_delivery)) |>
  mutate(pickup_zone = factor(pickup_zone, levels = zone_order)) |>
  ggplot(aes(pickup_zone, customer_rating_post_delivery, fill = pickup_zone)) +
  geom_boxplot(alpha = 0.85, outlier.alpha = 0.35) +
  labs(title = "Rating by pickup zone (ordered by median)",
       x = "Pickup zone", y = "Rating") +
  theme_minimal() + theme(legend.position = "none")

## 4 hypothesis tests

In [ ]:
# T1. Chi square pickup_zone vs delivery_status
tab_zone <- with(filter(dat, delivered), table(pickup_zone, delivery_status))
chi_zone <- chisq.test(tab_zone)
cat(sprintf("T1. pickup_zone vs delivery_status: X^2 = %.2f, df = %d, p = %.4f\n",
            chi_zone$statistic, chi_zone$parameter, chi_zone$p.value))

# T2. Chi square service_type vs delivery_status
tab_svc <- with(filter(dat, delivered), table(service_type, delivery_status))
chi_svc <- chisq.test(tab_svc)
cat(sprintf("T2. service_type vs delivery_status: X^2 = %.2f, df = %d, p = %.4f\n",
            chi_svc$statistic, chi_svc$parameter, chi_svc$p.value))

In [ ]:
# T3. Welch t-test on route distance for failed vs other.
fd <- dat |> filter(delivered) |> mutate(failed = delivery_status == "Failed")
t_dist <- t.test(route_distance_km ~ failed, data = fd, var.equal = FALSE)
cat(sprintf("T3. Failed vs other distance: t = %.2f, df = %.1f, p = %.4f (mean Failed=%.2f, other=%.2f)\n",
            t_dist$statistic, t_dist$parameter, t_dist$p.value,
            mean(fd$route_distance_km[fd$failed], na.rm = TRUE),
            mean(fd$route_distance_km[!fd$failed], na.rm = TRUE)))

# T4. Pearson correlation app_engagement vs rating.
cd <- dat |> filter(!is.na(app_engagement_score), !is.na(customer_rating_post_delivery))
ct <- cor.test(cd$app_engagement_score, cd$customer_rating_post_delivery)
cat(sprintf("T4. app_engagement vs rating: n = %d, r = %.3f, p = %.4f\n",
            nrow(cd), ct$estimate, ct$p.value))

## Logistic regression of delivery failure

In [ ]:
mdl <- dat |>
  filter(delivered) |>
  mutate(failed = as.integer(delivery_status == "Failed")) |>
  select(failed, service_type, pickup_zone, route_distance_km,
         loyalty_score, app_engagement_score, priority_level) |>
  drop_na()

cat("n in model:", nrow(mdl), "\n")
fit <- glm(failed ~ service_type + pickup_zone + route_distance_km +
                    loyalty_score + app_engagement_score + priority_level,
           data = mdl, family = binomial)
summary(fit)

In [ ]:
# Odds ratios with 95% confidence intervals.
or <- exp(coef(fit))
or_ci <- exp(confint.default(fit))
data.frame(term = names(or),
           odds_ratio = round(or, 3),
           ci_low = round(or_ci[, 1], 3),
           ci_high = round(or_ci[, 2], 3),
           row.names = NULL)

## What came out of the R run

Out of the 4 hypothesis tests, only the chi square on `pickup_zone` reaches the conventional 5 percent threshold, with p = 0.0103. The logistic regression then isolates `priority_levelCritical` as the only significant coefficient (odds ratio 0.275, 95 percent CI 0.082 to 0.929), with everything else non significant. So the operational signal is zone level rather than order level, with the smaller side finding that critical priority orders fail less often than the low priority baseline.